# Análise exploratória dados educação

In [ ]:
# Visualizando os arquivos de educacao

import pandas as pd
from pathlib import Path

pasta = Path("../data/raw/educacao")

for arquivo in pasta.rglob("*.csv"):
    print(arquivo.name)

In [ ]:
# Carregando tabela de cursos técnicos

cursos_tecnicos = pd.read_csv(
    '../data/raw/educacao/Censo Escolar 2025/Tabela_Curso_Tecnico_2025_V2.csv',
    sep=';',
    encoding='latin1',
    low_memory=False
)

In [ ]:
print("Dimensões:", cursos_tecnicos.shape)
print("Colunas:", cursos_tecnicos.columns.tolist())
display(cursos_tecnicos.head())

In [ ]:
# Identificar a granularidade dos dados
print("Instituições únicas:", cursos_tecnicos["CO_ENTIDADE"].nunique())
print("Cursos únicos:", cursos_tecnicos["CO_CURSO_EDUC_PROFISSIONAL"].nunique())
print("Municípios únicos:", cursos_tecnicos["CO_MUNICIPIO"].nunique())

print(
    "Combinações instituição-curso duplicadas:",
    cursos_tecnicos.duplicated(
        subset=["CO_ENTIDADE", "CO_CURSO_EDUC_PROFISSIONAL"]
    ).sum()
)

In [ ]:
colunas = [
    "CO_MUNICIPIO",
    "CO_ENTIDADE",
    "CO_CURSO_EDUC_PROFISSIONAL",
    "NO_AREA_CURSO_PROFISSIONAL",
    "QT_CURSO_TEC",
    "QT_MAT_CURSO_TEC"
]

cursos_tecnicos[colunas].isna().sum()

In [ ]:
print(cursos_tecnicos["QT_CURSO_TEC"].value_counts().sort_index())

In [ ]:
# Investigando as matrículas

cursos_tecnicos["QT_MAT_CURSO_TEC"].describe()

In [ ]:
# Investigando o valor extremo

cursos_tecnicos[
    cursos_tecnicos['QT_MAT_CURSO_TEC'] == cursos_tecnicos['QT_MAT_CURSO_TEC'].max()
][['NO_ENTIDADE', 'NO_MUNICIPIO', 'SG_UF', 'NO_CURSO_EDUC_PROFISSIONAL', 'QT_MAT_CURSO_TEC']]

In [ ]:
registro_max = cursos_tecnicos.loc[
    cursos_tecnicos["QT_MAT_CURSO_TEC"].idxmax()
]

display(registro_max)

In [ ]:
# Verificando a distribuição dos maiores cursos

cursos_tecnicos.nlargest(10, 'QT_MAT_CURSO_TEC')[['NO_ENTIDADE', 'NO_MUNICIPIO', 'SG_UF', 'NO_CURSO_EDUC_PROFISSIONAL', 'QT_MAT_CURSO_TEC']]

In [ ]:
# Medindo a concentração de matrículas 

(cursos_tecnicos.nlargest(10, 'QT_MAT_CURSO_TEC')['QT_MAT_CURSO_TEC'].sum()) / (cursos_tecnicos['QT_MAT_CURSO_TEC']).sum() * 100

# Os 10 maiores registros representam aproximadamente 2,78% do total nacional de matrículas técnicas em 2025.

# Investigando a tabela de turmas

In [ ]:
# Carregando a tabela com turmas 

turmas_2025 = pd.read_csv(
    '../data/raw/educacao/Censo Escolar 2025/Tabela_Turma_2025_V2.csv',
    sep=';',
    encoding='latin1',
    low_memory=False

)

In [ ]:
# Investigação inicial da tabela

print("Dimensões:", turmas_2025.shape)
print('Colunas:', list(turmas_2025.columns))
display(turmas_2025.head())

In [ ]:
# Investigar o ensino técnico EAD

print(
    turmas_2025[
        ["QT_TUR_PROF_TEC", "QT_TUR_PROF_TEC_EAD"]
    ].describe()
)

In [ ]:
# Calculando a participação do EAD

print("Total de turmas técnicas:", turmas_2025["QT_TUR_PROF_TEC"].sum())
print("Total de turmas técnicas EAD:", turmas_2025["QT_TUR_PROF_TEC_EAD"].sum())

In [ ]:
# Investigando as instituições com mais turmas EAD

turmas_2025.nlargest(10, 'QT_TUR_PROF_TEC_EAD')[['NO_ENTIDADE', 'NO_MUNICIPIO', 'SG_UF', 'QT_TUR_PROF_TEC', 'QT_TUR_PROF_TEC_EAD']]

In [ ]:
# Verificando a instituição 'INSTITUTO MONITOR' 

turmas_2025[
    turmas_2025['CO_ENTIDADE'] == 35257916
][['NO_ENTIDADE', 'NO_MUNICIPIO', 'SG_UF', 'QT_TUR_PROF_TEC', 'QT_TUR_PROF_TEC_EAD']]

Agora temos uma evidência importante: o Instituto Monitor registra todas as suas turmas técnicas como EAD.

Isso reforça nossa preocupação com as 16.254 matrículas em Transações Imobiliárias identificadas anteriormente.

Entretanto, ainda não podemos afirmar que todas essas matrículas são EAD, pois a tabela de turmas apresenta informações agregadas por instituição, não por curso específico.

Decisão metodológica provisória: manter as matrículas como indicador de volume educacional associado à instituição, mas não interpretá-las diretamente como demanda ou oferta presencial do município.

In [ ]:
# Investigando cursos do 'INSTITUTO MONITOR'

print("Número de cursos oferecidos:" , cursos_tecnicos[
    cursos_tecnicos['CO_ENTIDADE'] == 35257916
]['NO_CURSO_EDUC_PROFISSIONAL'].nunique())

print("Total de matrículas:" , cursos_tecnicos[
    cursos_tecnicos['CO_ENTIDADE'] == 35257916
]['QT_MAT_CURSO_TEC'].sum())

In [ ]:
# Investigando a oferta EAD por instituição 

print("Instituições únicas:", turmas_2025["CO_ENTIDADE"].nunique())
print("Instituições duplicadas:", turmas_2025["CO_ENTIDADE"].duplicated().sum())

In [ ]:
(turmas_2025['QT_TUR_PROF_TEC'] > 0).sum()

In [ ]:
exclusiva_ead = (turmas_2025["QT_TUR_PROF_TEC"] > 0) & (turmas_2025["QT_TUR_PROF_TEC"] == turmas_2025["QT_TUR_PROF_TEC_EAD"])
print("Instituições (exclusiva_ead):", exclusiva_ead.sum())
display(
    turmas_2025.loc[exclusiva_ead, [
        "NO_ENTIDADE", "NO_MUNICIPIO", "SG_UF",
        "QT_TUR_PROF_TEC", "QT_TUR_PROF_TEC_EAD",
    ]].sort_values("QT_TUR_PROF_TEC", ascending=False).head(10)
)

In [ ]:
oferta_mista = (turmas_2025["QT_TUR_PROF_TEC"] > 0) & (turmas_2025["QT_TUR_PROF_TEC_EAD"] > 0) & (turmas_2025["QT_TUR_PROF_TEC_EAD"] < turmas_2025["QT_TUR_PROF_TEC"])
print("Instituições (oferta_mista):", oferta_mista.sum())
display(
    turmas_2025.loc[oferta_mista, [
        "NO_ENTIDADE", "NO_MUNICIPIO", "SG_UF",
        "QT_TUR_PROF_TEC", "QT_TUR_PROF_TEC_EAD",
    ]].sort_values("QT_TUR_PROF_TEC", ascending=False).head(10)
)

In [ ]:
exclusiva_presencial = (turmas_2025["QT_TUR_PROF_TEC"] > 0) & (turmas_2025["QT_TUR_PROF_TEC_EAD"] == 0)
print("Instituições (exclusiva_presencial):", exclusiva_presencial.sum())
display(
    turmas_2025.loc[exclusiva_presencial, [
        "NO_ENTIDADE", "NO_MUNICIPIO", "SG_UF",
        "QT_TUR_PROF_TEC", "QT_TUR_PROF_TEC_EAD",
    ]].sort_values("QT_TUR_PROF_TEC", ascending=False).head(10)
)

Temos 211 instituições que não se encaixaram nas três categorias.
Uma hipótese é que existam registros em que QT_TUR_PROF_TEC_EAD seja maior que QT_TUR_PROF_TEC.

In [ ]:
print(
    "Instituições com turmas EAD superiores ao total:",
    (
        turmas_2025["QT_TUR_PROF_TEC_EAD"]
        > turmas_2025["QT_TUR_PROF_TEC"]
    ).sum()
)

In [ ]:
print((
    (turmas_2025["QT_TUR_PROF_TEC"] > 0) &
    (turmas_2025["QT_TUR_PROF_TEC_EAD"].isna())
).sum())

In [ ]:
# Verificando a cobertura municipal

turmas_2025[
    turmas_2025['QT_TUR_PROF_TEC'] > 0
]['CO_MUNICIPIO'].nunique()

In [ ]:
# Comparando a cobertura geográfica 
# Vamos descobrir quantos municípios aparecem na tabela de cursos técnicos, mas não na tabela de turmas com oferta técnica.

municipios_com_turmas = turmas_2025[
    turmas_2025['QT_TUR_PROF_TEC'] > 0
]['CO_MUNICIPIO']

cursos_tecnicos[~cursos_tecnicos['CO_MUNICIPIO'].isin(municipios_com_turmas)]['CO_MUNICIPIO'].nunique()

In [ ]:
# Verificando municípios exclusivos da tabela de turmas 

municipios_com_turmas[~municipios_com_turmas.isin(cursos_tecnicos['CO_MUNICIPIO'])].nunique()

In [ ]:
# Investigando os 41 municípios que aparecem apenas na tabela cursos_tecnicos

cursos_tecnicos[
    ~cursos_tecnicos["CO_MUNICIPIO"].isin(municipios_com_turmas)
][['NO_MUNICIPIO', 'SG_UF', 'NO_ENTIDADE', 'NO_CURSO_EDUC_PROFISSIONAL', 'QT_MAT_CURSO_TEC']].head(10)

Decisão metodológica
Para o projeto UNIESF, recomendo:
- Tabela de cursos: fonte principal para identificar oferta de cursos, áreas de formação e matrículas.
- Tabela de turmas: fonte complementar para identificar a presença de EAD nas instituições.
- 41 municípios divergentes: preservar na análise, registrando a inconsistência como limitação.

In [ ]:
educacao_municipal_2025 = cursos_tecnicos.groupby("CO_MUNICIPIO").agg(
    instituicoes_tecnicas=("CO_ENTIDADE", "nunique"),
     ofertas_cursos=("CO_CURSO_EDUC_PROFISSIONAL", "size"),
    cursos_distintos=('NO_CURSO_EDUC_PROFISSIONAL', 'nunique'),
    matriculas_tecnicas=('QT_MAT_CURSO_TEC', 'sum')
).reset_index()

display(educacao_municipal_2025)

In [ ]:
print("Dimensões:", educacao_municipal_2025.shape)
display(educacao_municipal_2025.head())

### Integrar a modalidade EAD

In [ ]:
turmas_municipais_2025 = turmas_2025.groupby('CO_MUNICIPIO').agg(
    turmas_tecnicas=('QT_TUR_PROF_TEC', 'sum'),
    turmas_tecnicas_ead=('QT_TUR_PROF_TEC_EAD', 'sum')
).reset_index()

display(turmas_municipais_2025.head())

In [ ]:
# validação 

print("Dimensões:", turmas_municipais_2025.shape)

print(
    turmas_municipais_2025[
        ["turmas_tecnicas", "turmas_tecnicas_ead"]
    ].sum()
)

print(
    "Municípios duplicados:",
    turmas_municipais_2025["CO_MUNICIPIO"].duplicated().sum()
)

In [ ]:
# Integrando os indicadores municipais

educacao_municipal_2025 = educacao_municipal_2025.merge(
    turmas_municipais_2025,
    on="CO_MUNICIPIO",
    how="left",
    validate="one_to_one"
)

print("Dimensões:", educacao_municipal_2025.shape)
print("Valores ausentes:")
print(educacao_municipal_2025.isna().sum())
display(educacao_municipal_2025.head())

In [ ]:
import numpy as np

In [ ]:
educacao_municipal_2025['percentual_ead'] = np.where(
    educacao_municipal_2025['turmas_tecnicas'] > 0,
    ((educacao_municipal_2025['turmas_tecnicas_ead'] / educacao_municipal_2025['turmas_tecnicas'] * 100).round(2)), 
    np.nan)

display(educacao_municipal_2025.head())

In [ ]:
print("Percentuais ausentes:",
      educacao_municipal_2025["percentual_ead"].isna().sum())

print("Percentuais acima de 100%:",
      (educacao_municipal_2025["percentual_ead"] > 100).sum())

display(educacao_municipal_2025["percentual_ead"].describe())

In [ ]:
# Municípios com maior número absoluto de turmas técnicas EAD
display(
    educacao_municipal_2025
    .sort_values("turmas_tecnicas_ead", ascending=False)
    .head(10)
)

# Municípios com maior participação EAD, considerando ao menos 20 turmas técnicas
display(
    educacao_municipal_2025[
        educacao_municipal_2025["turmas_tecnicas"] >= 20
    ]
    .sort_values("percentual_ead", ascending=False)
    .head(10)
)

Preparar os indicadores educacionais para integração

In [ ]:
# Carrega a referência oficial dos municípios brasileiros
ibge_municipios = pd.read_parquet(
    "../data/processed/municipios_ibge.parquet"
)

# Associa nomes e UFs aos indicadores educacionais
educacao_municipal_2025 = educacao_municipal_2025.merge(
    ibge_municipios[["codigo_ibge", "municipio", "uf"]],
    left_on="CO_MUNICIPIO",
    right_on="codigo_ibge",
    how="left",
    validate="one_to_one"
)

# Confere a integridade do cruzamento
print("Dimensões:", educacao_municipal_2025.shape)
print("Municípios sem correspondência:",
      educacao_municipal_2025["municipio"].isna().sum())

In [ ]:
# Remove o código municipal redundante, mantendo o código IBGE padronizado
educacao_municipal_2025 = educacao_municipal_2025.drop(
    columns=["CO_MUNICIPIO"]
)

pasta_destino = Path("../data/processed/educacao")
pasta_destino.mkdir(parents=True, exist_ok=True)


# Salva os indicadores educacionais municipais em formato Parquet
educacao_municipal_2025.to_parquet(
    "../data/processed/educacao/indicadores_educacao_municipal_2025.parquet",
    index=False
)

### Quais áreas de formação são oferecidas?
Vamos utilizar novamente cursos_tecnicos para identificar quais áreas de formação profissional possuem mais ofertas e matrículas.

In [ ]:
# Agrupa os cursos técnicos por área de formação profissional
oferta_por_area_2025 = cursos_tecnicos.groupby(
    "NO_AREA_CURSO_PROFISSIONAL"
).agg(
    ofertas=("CO_CURSO_EDUC_PROFISSIONAL", "size"),
    cursos_distintos=("CO_CURSO_EDUC_PROFISSIONAL", "nunique"),
    instituicoes=("CO_ENTIDADE", "nunique"),
    matriculas=("QT_MAT_CURSO_TEC", "sum")
).reset_index()

# Ordena as áreas pelo maior número de ofertas
oferta_por_area_2025 = oferta_por_area_2025.sort_values(
    "ofertas", ascending=False
)

display(oferta_por_area_2025)

Gestão e negócios lidera em ofertas e matrículas, enquanto Ambiente e saúde possui mais matrículas que Informação e comunicação, apesar de ter menos ofertas.
O que isso significa para a UNIESF?
Já podemos começar a relacionar a oferta educacional com as atividades econômicas que analisamos no CAGED.

Por exemplo:
- Indústria de transformação: investigar cursos em Controle e processos industriais e Produção alimentícia.
- Comércio: investigar cursos em Gestão e negócios.
- Serviços de alimentação: investigar cursos em Turismo, hospitalidade e lazer e Produção alimentícia.

### Identificando cursos específicos

In [ ]:
# Agrega ofertas e matrículas por curso técnico específico

oferta_por_curso_2025 = cursos_tecnicos.groupby(
    ["CO_CURSO_EDUC_PROFISSIONAL", "NO_CURSO_EDUC_PROFISSIONAL",
     "NO_AREA_CURSO_PROFISSIONAL"]
).agg(
    ofertas=("CO_ENTIDADE", "size"),
    municipios=("CO_MUNICIPIO", "nunique"),
    matriculas=("QT_MAT_CURSO_TEC", "sum")
).reset_index()

# Exibe os 15 cursos com maior número de ofertas

display(
    oferta_por_curso_2025
    .sort_values("ofertas", ascending=False)
    .head(15)
)

Relação com o CAGED
Temos algumas conexões interessantes para investigar:
- Administração e Logística: possíveis relações com atividades administrativas, comércio e logística.
- Enfermagem: relação com o setor de saúde.
- Eletrotécnica e Mecânica: possíveis relações com a indústria de transformação.
- Agropecuária: relação com atividades agrícolas e pecuárias.

### Analisar a oferta educacional por território

In [ ]:
educacao_municipio_curso_2025 = cursos_tecnicos.groupby(
    ['CO_MUNICIPIO', 'CO_CURSO_EDUC_PROFISSIONAL','NO_CURSO_EDUC_PROFISSIONAL', 'NO_AREA_CURSO_PROFISSIONAL' ]).agg(
        instituicoes=('CO_ENTIDADE', 'nunique'),
        matriculas=('QT_MAT_CURSO_TEC', 'sum')
    ).reset_index()

display(educacao_municipio_curso_2025.head(10))

In [ ]:
# Verifica as dimensões da tabela agregada
print("Dimensões:", educacao_municipio_curso_2025.shape)

# Confere se o total de matrículas foi preservado
print("Matrículas originais:", cursos_tecnicos["QT_MAT_CURSO_TEC"].sum())
print("Matrículas agregadas:", educacao_municipio_curso_2025["matriculas"].sum())

# Verifica possíveis duplicações na combinação município-curso
print("Combinações duplicadas:",
      educacao_municipio_curso_2025.duplicated(
          subset=["CO_MUNICIPIO", "CO_CURSO_EDUC_PROFISSIONAL"]
      ).sum())

In [ ]:
# Padroniza o nome da chave municipal para integração com outras bases
educacao_municipio_curso_2025 = educacao_municipio_curso_2025.rename(
    columns={"CO_MUNICIPIO": "codigo_ibge"}
)

# Salva a tabela detalhada por município e curso técnico
educacao_municipio_curso_2025.to_parquet(
    "../data/processed/educacao/educacao_municipio_curso_2025.parquet",
    index=False
)

## Investigar a relação entre cursos e ocupações
#### Para o módulo “O que oferecer?”, vamos priorizar a relação entre:
- CBO: quais ocupações estão contratando migrantes.
- INEP: quais cursos técnicos existem nos municípios.
- CNAE: quais setores econômicos concentram essas contratações.
A CBO será nossa referência inicial, porque descreve ocupações, enquanto a CNAE descreve atividades econômicas das empresas.

## Análise dos microdados do Ensino Superior 2024 (INEP)

In [ ]:
# Carrega os cadastros de cursos e instituições de ensino superior
cursos_superior = pd.read_csv(
    "../data/raw/educacao/INEP Ensino Superior/MICRODADOS_CADASTRO_CURSOS_2024.CSV",
    sep=";",
    encoding="latin-1",
    low_memory=False
)

instituicoes_superior = pd.read_csv(
    "../data/raw/educacao/INEP Ensino Superior/MICRODADOS_ED_SUP_IES_2024.CSV",
    sep=";",
    encoding="latin-1",
    low_memory=False
)

# Confere dimensões e identifica colunas relevantes para a análise
print("Cursos superiores:", cursos_superior.shape)
print("Instituições:", instituicoes_superior.shape)

for nome, df in [
    ("CURSOS", cursos_superior),
    ("INSTITUIÇÕES", instituicoes_superior)
]:
    print(f"\n{nome} — Colunas relevantes:")

    print([
        coluna for coluna in df.columns
        if any(termo in coluna.upper() for termo in [
            "MUNICIPIO", "CURSO", "IES", "MODALIDADE",
            "AREA", "MATRIC", "UF"
        ])
    ])

In [ ]:
# Verifica a quantidade de instituições, cursos e municípios distintos
for coluna in ["CO_IES", "CO_CURSO", "CO_MUNICIPIO"]:
    print(f"{coluna}: {cursos_superior[coluna].nunique():,}")

# Verifica se existem repetições na combinação curso-município
print(
    "Duplicatas curso-município:",
    cursos_superior.duplicated(
        subset=["CO_CURSO", "CO_MUNICIPIO"]
    ).sum()
)

# Identifica os códigos das modalidades de ensino
print("\nModalidades de ensino:")
display(cursos_superior["TP_MODALIDADE_ENSINO"].value_counts(dropna=False))

# Investiga o significado e a distribuição de QT_CURSO
print("\nQuantidade de cursos por registro:")
display(cursos_superior["QT_CURSO"].value_counts(dropna=False).head(10))

# Verifica quais municípios aparecem na tabela de instituições
print(
    "\nMunicípios com sede de IES:",
    instituicoes_superior["CO_MUNICIPIO_IES"].nunique()
)

In [ ]:
# Confere se a coluna de dimensão existe
print("TP_DIMENSAO" in cursos_superior.columns)

# Conta os registros por dimensão e modalidade
display(
    cursos_superior.groupby(
        ["TP_DIMENSAO", "TP_MODALIDADE_ENSINO"],
        dropna=False
    ).agg(
        registros=("CO_CURSO", "size"),
        cursos_contabilizados=("QT_CURSO", "sum"),
        municipios=("CO_MUNICIPIO", "nunique")
    ).reset_index()
)

In [ ]:
# Localiza as variáveis de matrículas, ingressantes e concluintes
colunas_alunos = [
    coluna for coluna in cursos_superior.columns
    if coluna.startswith(("QT_MAT", "QT_ING", "QT_CONC"))
]

# Exibe as variáveis disponíveis
print(colunas_alunos)

In [ ]:
# Seleciona os indicadores educacionais relevantes
indicadores_superior = [
    "QT_MAT",
    "QT_ING",
    "QT_CONC",
    "QT_MAT_NACESTRANG"
]

# Resume matrículas, ingressantes e concluintes por dimensão
resumo_dimensoes = cursos_superior.groupby(
    ["TP_DIMENSAO", "TP_MODALIDADE_ENSINO"]
)[indicadores_superior].sum(min_count=1).reset_index()

# Exibe os totais por dimensão e modalidade
display(resumo_dimensoes)

# Verifica valores ausentes nos indicadores
display(
    cursos_superior[indicadores_superior]
    .isna().sum()
    .to_frame("valores_ausentes")
)

In [ ]:
# Separa os registros presenciais e EAD por município
superior_presencial = cursos_superior[
    cursos_superior["TP_DIMENSAO"] == 1
]

superior_ead = cursos_superior[
    cursos_superior["TP_DIMENSAO"] == 2
]

# Agrega indicadores da oferta presencial por município
presencial_municipal_2024 = superior_presencial.groupby(
    "CO_MUNICIPIO"
).agg(
    cursos_presenciais=("QT_CURSO", "sum"),
    ies_presenciais=("CO_IES", "nunique"),
    matriculas_presenciais=("QT_MAT", "sum"),
    matriculas_estrangeiros_presencial=("QT_MAT_NACESTRANG", "sum")
).reset_index()

# Agrega as matrículas EAD por município do polo
ead_municipal_2024 = superior_ead.groupby(
    "CO_MUNICIPIO"
).agg(
    matriculas_ead=("QT_MAT", "sum"),
    ingressantes_ead=("QT_ING", "sum"),
    matriculas_estrangeiros_ead=("QT_MAT_NACESTRANG", "sum")
).reset_index()

# Confere a cobertura municipal e os totais
print("Municípios presenciais:", len(presencial_municipal_2024))
print("Municípios EAD:", len(ead_municipal_2024))

print("Cursos presenciais:",
      presencial_municipal_2024["cursos_presenciais"].sum())

print("Matrículas EAD:",
      ead_municipal_2024["matriculas_ead"].sum())

In [ ]:
# Identifica registros EAD sem código de município
ead_sem_municipio = superior_ead[
    superior_ead["CO_MUNICIPIO"].isna()
]

# Verifica quantidade de registros e matrículas afetadas
print("Registros sem município:", len(ead_sem_municipio))
print("Matrículas sem município:", ead_sem_municipio["QT_MAT"].sum())

# Confere se a diferença corresponde às matrículas sem município
print(
    "Diferença entre total e agregação:",
    superior_ead["QT_MAT"].sum()
    - ead_municipal_2024["matriculas_ead"].sum()
)

# Examina os registros sem município
display(
    ead_sem_municipio[
        ["CO_IES", "CO_CURSO", "CO_MUNICIPIO", "QT_MAT"]
    ].head(10)
)

In [ ]:
# Integra os indicadores presenciais e EAD por município
educacao_superior_municipal_2024 = presencial_municipal_2024.merge(
    ead_municipal_2024,
    on="CO_MUNICIPIO",
    how="outer",
    validate="one_to_one",
    indicator=True
)

# Padroniza a chave territorial para os demais arquivos do projeto
educacao_superior_municipal_2024 = educacao_superior_municipal_2024.rename(
    columns={"CO_MUNICIPIO": "codigo_ibge"}
)

# Verifica a cobertura de cada modalidade
print("Dimensões:", educacao_superior_municipal_2024.shape)

print("\nCobertura:")
print(educacao_superior_municipal_2024["_merge"].value_counts())

# Confere a ausência de municípios duplicados
print(
    "\nCódigos IBGE duplicados:",
    educacao_superior_municipal_2024["codigo_ibge"].duplicated().sum()
)

# Verifica se os totais de matrículas foram preservados
print(
    "\nMatrículas presenciais:",
    educacao_superior_municipal_2024["matriculas_presenciais"].sum()
)

print(
    "Matrículas EAD:",
    educacao_superior_municipal_2024["matriculas_ead"].sum()
)

In [ ]:
# Carrega a referência oficial dos municípios
municipios_ibge = ibge_municipios

# Associa nomes e UFs aos indicadores de ensino superior
educacao_superior_municipal_2024 = (
    educacao_superior_municipal_2024
    .drop(columns=["_merge"])
    .merge(
        municipios_ibge[["codigo_ibge", "municipio", "uf"]],
        on="codigo_ibge",
        how="left",
        validate="one_to_one"
    )
)

# Verifica correspondências geográficas e duplicações
print("Dimensões:", educacao_superior_municipal_2024.shape)
print("Municípios sem nome:",
      educacao_superior_municipal_2024["municipio"].isna().sum())
print("Municípios sem UF:",
      educacao_superior_municipal_2024["uf"].isna().sum())
print("Códigos duplicados:",
      educacao_superior_municipal_2024["codigo_ibge"].duplicated().sum())

In [ ]:
# Define o caminho do arquivo processado
arquivo_saida = (
    "../data/processed/educacao/"
    "indicadores_ensino_superior_municipal_2024.parquet"
)

# Exporta os indicadores municipais de ensino superior
educacao_superior_municipal_2024.to_parquet(
    arquivo_saida,
    index=False
)

# Confirma o salvamento
print("Arquivo salvo:", arquivo_saida)